# 🧠 Ejercicio: Red Neuronal con Keras/TensorFlow

Una **Red Neuronal Artificial (RNA)** es un modelo de aprendizaje supervisado inspirado en el cerebro humano. Está formada por capas de **neuronas** conectadas que aprenden patrones a través de un proceso llamado **backpropagation**.

## 📌 Objetivo del ejercicio
Predecir si un paciente tiene **diabetes** (clasificación binaria) usando el dataset **Pima Indians Diabetes**, con una red neuronal totalmente conectada (MLP).

---

## Arquitectura que construiremos
```
Entrada (8 features)
    ↓
[Dense 32, ReLU] → Dropout 20%
    ↓
[Dense 16, ReLU] → Dropout 20%
    ↓
[Dense 1, Sigmoid]  ← salida: probabilidad de diabetes
```

## Pasos
1. Importar librerías
2. Cargar y explorar datos
3. Preprocesar (escalar + dividir)
4. Construir la red neuronal
5. Entrenar el modelo
6. Evaluar resultados
7. Visualizar curvas de aprendizaje
8. Matriz de confusión y métricas
9. Predicción sobre nuevos pacientes

## Paso 1 — Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print(f'✅ TensorFlow {tf.__version__} cargado')
print(f'✅ Keras {keras.__version__} cargado')

## Paso 2 — Cargar y explorar datos

Usamos el dataset **Pima Indians Diabetes** (768 pacientes, 8 variables clínicas, etiqueta binaria: 0=sano, 1=diabético).

In [ ]:
# Dataset disponible directamente desde URL
url = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df = pd.read_csv(url)

print(f'Forma del dataset: {df.shape}')
print(f'\nColumnas: {list(df.columns)}')
df.head()

In [ ]:
print('=== Estadísticas descriptivas ===')
print(df.describe().round(2))

print(f'\n=== Distribución de clases ===')
conteo = df['Outcome'].value_counts()
print(f'  Sanos     (0): {conteo[0]} ({conteo[0]/len(df)*100:.1f}%)')
print(f'  Diabéticos(1): {conteo[1]} ({conteo[1]/len(df)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
features = [c for c in df.columns if c != 'Outcome']
colores = ['#3498DB', '#E74C3C']

for i, feat in enumerate(features):
    ax = axes[i // 4][i % 4]
    for clase, color in zip([0, 1], colores):
        ax.hist(df[df['Outcome'] == clase][feat], bins=20,
                alpha=0.6, color=color,
                label='Sano' if clase == 0 else 'Diabético')
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Distribución de variables por clase', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Paso 3 — Preprocesamiento

In [ ]:
# Separar features y target
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

# Dividir: 70% entrenamiento, 15% validación, 15% prueba
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

# Escalar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f'Entrenamiento : {X_train.shape[0]} muestras')
print(f'Validación    : {X_val.shape[0]} muestras')
print(f'Prueba        : {X_test.shape[0]} muestras')
print(f'\nFeatures de entrada: {X_train.shape[1]}')

## Paso 4 — Construir la Red Neuronal

Usamos la API **Sequential** de Keras. Cada capa `Dense` aplica la transformación: `salida = activacion(W·x + b)`

| Capa | Neuronas | Activación | Propósito |
|------|----------|------------|-----------|
| Dense 1 | 32 | ReLU | Extrae patrones generales |
| Dropout | 20% | — | Regularización (evita overfitting) |
| Dense 2 | 16 | ReLU | Refina representaciones |
| Dropout | 20% | — | Regularización |
| Salida | 1 | Sigmoid | Probabilidad [0, 1] |

In [ ]:
def construir_modelo(input_dim):
    modelo = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),

        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),

        layers.Dense(1, activation='sigmoid')
    ], name='Red_Diabetes')

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

modelo = construir_modelo(X_train.shape[1])
modelo.summary()

## Paso 5 — Entrenar el modelo

Usamos **EarlyStopping** para detener el entrenamiento si la validación deja de mejorar (evita overfitting).

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

historia = modelo.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Entrenamiento completado en {len(historia.history["loss"])} épocas')

## Paso 6 — Curvas de aprendizaje

In [ ]:
h = historia.history
epocas = range(1, len(h['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pérdida
axes[0].plot(epocas, h['loss'],     color='#3498DB', linewidth=2, label='Entrenamiento')
axes[0].plot(epocas, h['val_loss'], color='#E74C3C', linewidth=2, linestyle='--', label='Validación')
axes[0].set_title('Pérdida (Binary Crossentropy)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Exactitud
axes[1].plot(epocas, h['accuracy'],     color='#2ECC71', linewidth=2, label='Entrenamiento')
axes[1].plot(epocas, h['val_accuracy'], color='#E67E22', linewidth=2, linestyle='--', label='Validación')
axes[1].set_title('Exactitud (Accuracy)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Paso 7 — Evaluación en conjunto de prueba

In [ ]:
loss_test, acc_test = modelo.evaluate(X_test, y_test, verbose=0)
print(f'📊 Resultados en conjunto de PRUEBA (datos nunca vistos):')
print(f'   Loss     : {loss_test:.4f}')
print(f'   Accuracy : {acc_test:.4f}  ({acc_test*100:.1f}%)')

# Predicciones
y_prob = modelo.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print(f'\n=== Reporte de Clasificación ===')
print(classification_report(y_test, y_pred, target_names=['Sano', 'Diabético']))

auc = roc_auc_score(y_test, y_prob)
print(f'AUC-ROC: {auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Sano', 'Diabético'],
            yticklabels=['Sano', 'Diabético'],
            linewidths=1, linecolor='white')
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicción')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#3498DB', linewidth=2.5, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Clasificador aleatorio')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#3498DB')
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Paso 8 — Importancia de variables (análisis de pesos)

In [ ]:
# Extraer pesos de la primera capa
pesos_capa1 = np.abs(modelo.layers[0].get_weights()[0])  # shape (8, 32)
importancia = pesos_capa1.mean(axis=1)  # promedio por feature
importancia = importancia / importancia.sum()  # normalizar

feature_names = [c for c in df.columns if c != 'Outcome']
orden = np.argsort(importancia)[::-1]

plt.figure(figsize=(10, 5))
colores_bar = ['#E74C3C' if i == orden[0] else '#3498DB' for i in range(len(feature_names))]
plt.bar([feature_names[i] for i in orden],
        [importancia[i] for i in orden],
        color=[colores_bar[i] for i in orden],
        edgecolor='white', linewidth=0.5)
plt.title('Importancia estimada de variables (pesos capa 1)', fontsize=13, fontweight='bold')
plt.ylabel('Importancia relativa')
plt.xticks(rotation=20, ha='right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Paso 9 — Predecir sobre nuevos pacientes

In [ ]:
# Columnas: Pregnancies, Glucose, BloodPressure, SkinThickness,
#           Insulin, BMI, DiabetesPedigreeFunction, Age

nuevos_pacientes = np.array([
    [2,  85,  66, 29,  0, 26.6, 0.351, 31],  # Paciente A — perfil bajo riesgo
    [8, 183,  64,  0,  0, 23.3, 0.672, 32],  # Paciente B — glucosa alta
    [6, 148,  72, 35,  0, 33.6, 0.627, 50],  # Paciente C — múltiples factores
])

nuevos_escalados = scaler.transform(nuevos_pacientes)
probabilidades   = modelo.predict(nuevos_escalados, verbose=0).flatten()
predicciones     = (probabilidades >= 0.5).astype(int)

etiquetas = ['Sano ✅', 'Diabético ⚠️']
print('=== Predicciones para nuevos pacientes ===')
for i, (prob, pred) in enumerate(zip(probabilidades, predicciones)):
    print(f'  Paciente {chr(65+i)}: {etiquetas[pred]}  (probabilidad: {prob:.1%})')

## ✅ Resumen de conceptos clave

| Concepto | Descripción |
|---|---|
| **Neurona** | Aplica `activación(W·x + b)` a sus entradas |
| **ReLU** | `max(0, x)` — activa solo valores positivos, evita el problema del gradiente |
| **Sigmoid** | Salida entre 0 y 1 — ideal para clasificación binaria |
| **Dropout** | Apaga neuronas aleatoriamente durante entrenamiento → regularización |
| **Adam** | Optimizador adaptativo, mezcla de Momentum + RMSProp |
| **Binary Crossentropy** | Función de pérdida para clasificación binaria |
| **EarlyStopping** | Para el entrenamiento cuando la validación deja de mejorar |
| **AUC-ROC** | Mide la capacidad discriminativa del modelo (1.0 = perfecto) |

---

## 🧩 Retos adicionales
1. Modifica la arquitectura: agrega una capa Dense(64) al inicio — ¿mejora?
2. Cambia la tasa de Dropout a 0.4 — ¿cómo afecta el overfitting?
3. Prueba el optimizador `SGD` en lugar de `Adam` — ¿cuántas épocas necesita?
4. Implementa `class_weight` para manejar el desbalance de clases
5. Guarda y recarga el modelo: `modelo.save('modelo_diabetes.keras')`